In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 6.6 The Pauli Matrices, Incompatible Observables, and the Uncertainty Relation

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VI — Quantum Mechanics",
    number="6.6",
    title="The Pauli Matrices, Incompatible Observables, and the Uncertainty Relation",
    blurb="Why looking along x erases what we knew along z. The three spin "
    "components do not commute, and that single algebraic fact is the whole of the "
    "uncertainty principle: when two observables fail to commute, no state makes "
    "both sharp, and the product of their uncertainties is bounded below by their "
    "commutator. We build the Pauli algebra, derive the uncertainty relation from "
    "Cauchy–Schwarz, and find the states that meet it exactly.",
    difficulty="advanced",
    estimate="150–185 min",
)

## Notebook overview

The Stern–Gerlach experiment ([§6.4](stern-gerlach-qubit.ipynb)) showed us, as a brute fact, that measuring spin along $x$ destroys
what we knew about spin along $z$; the postulates ([§6.5](postulates.ipynb)) gave us the variance $\Delta A$ that measures
how *un*-sharp an observable is in a state. This notebook supplies the missing piece — the **algebra**
that explains *why* the destruction happens and *how much* it must — and it turns out to be one of the
cleanest results in all of physics: the uncertainty principle is a three-line theorem about operators
that do not commute.

The central new object is the **commutator** $[A,B]=AB-BA$. We met it in [§6.2](operators-spectral-theorem.ipynb) as the test for whether
two operators share an eigenbasis; here it becomes the protagonist. Two observables are *compatible* —
can be simultaneously sharp in a common basis — exactly when they commute, and *incompatible* when they do not. The
**Pauli matrices** are the canonical incompatible observables: the three spin components pairwise fail
to commute, $[\sigma_x,\sigma_y]=2i\sigma_z$ and cyclically, so a state perfectly definite along one
axis is *maximally* uncertain along the others. That is the sequential Stern–Gerlach experiment, now
as algebra rather than anecdote.

The heart of the notebook is the **derivation** of the Robertson uncertainty relation $\Delta A\,
\Delta B\ge\tfrac12|\langle[A,B]\rangle|$. We do not assert it — we derive it, gently and in full,
from the **Cauchy–Schwarz inequality** of [§6.1](complex-vector-spaces.ipynb), the very inequality that bounded probability amplitudes
there. The lesson is worth stating plainly and is easy to get wrong: the uncertainty principle is
**not** a statement about clumsy instruments disturbing delicate systems. It is a theorem about
non-commuting Hermitian operators, true of *every* state whether or not anyone is looking, and it is
*saturated* — met with equality — by special minimum-uncertainty states.

As in every Volume VI notebook, each exercise opens with a **crystal-clear statement** and enumerated parts, each naming the exact operation — explicit Pauli-matrix construction, `commutator` and
`anticommutator` via the matrix-multiply `@`, `numpy.vdot` for the moments $\langle A\rangle$ and
$\langle A^2\rangle$, and `numpy.linalg.eigh` where spectra are needed.

> **How to read the checks.** Each exercise closes with a `validate` call: the full Pauli algebra
> (commutators, anticommutators, the product identity); the maximal uncertainty of $\sigma_x$ in a
> $\sigma_z$-eigenstate; the Robertson inequality holding for random observables and states; its
> saturation by $|{+}z\rangle$; and the simultaneous definiteness of commuting observables. A ✓ is
> strong evidence; a ✗ is a prompt to *locate the discrepancy*.
>
> **Conventions and scope.** We work with $\hbar=1$, so the spin operators are $S=\tfrac12\boldsymbol
> \sigma$ with eigenvalues $\pm\tfrac12$ (physically $\pm\hbar/2$); we restore $\hbar$ in formulas
> where it clarifies. The qubit is the running example, with general Hermitian observables on
> $\mathbb{C}^3$ for the general check. The position–momentum relation $\Delta x\,\Delta p\ge\hbar/2$
> from $[x,p]=i\hbar$ is [§6.9](position-representation.ipynb); the angular-momentum algebra $[J_i,J_j]=i\hbar\varepsilon_{ijk}J_k$ is
> [§6.14](angular-momentum-algebra.ipynb); complete sets of commuting observables catalogue the hydrogen atom in [§6.17](hydrogen-atom.ipynb). See Sakurai &
> Napolitano (§1.4); Robertson (1929); Schrödinger (1930); and Notebooks [§6.1](complex-vector-spaces.ipynb) (Cauchy–Schwarz), [§6.2](operators-spectral-theorem.ipynb)
> (commutators), [§6.4](stern-gerlach-qubit.ipynb) (the experiment), [§6.5](postulates.ipynb) (variance).

## Theory in brief

### The Pauli matrices and spin operators

The three **Pauli matrices** are the Hermitian observables of the qubit; the spin operators are
$S=\tfrac{\hbar}{2}\boldsymbol\sigma$,

```{math}
:label: eq-pauli
\sigma_x=\begin{pmatrix}0&1\\1&0\end{pmatrix},\ \sigma_y=\begin{pmatrix}0&-i\\i&0\end{pmatrix},\ \sigma_z=\begin{pmatrix}1&0\\0&-1\end{pmatrix},\qquad \sigma_i=\sigma_i^{\dagger},\ \sigma_i^2=I,\ \text{eigenvalues }\pm1 .
```

They are exactly the observables a Stern–Gerlach magnet oriented along $x$, $y$, $z$ measures ([§6.4](stern-gerlach-qubit.ipynb)).

### The algebra: commutators and the product identity

The **commutator** $[A,B]=AB-BA$ measures the failure to commute. The Pauli matrices satisfy

```{math}
:label: eq-pauli-algebra
[\sigma_x,\sigma_y]=2i\sigma_z\ (\text{cyclic}),\quad \{\sigma_i,\sigma_j\}=0\ (i\ne j),\quad \sigma_i\sigma_j=\delta_{ij}I+i\varepsilon_{ijk}\sigma_k ,
```

equivalently $[S_x,S_y]=i\hbar S_z$ — the **angular-momentum algebra** (generalized in [§6.14](angular-momentum-algebra.ipynb)). The
commutator is zero for compatible observables and non-zero for incompatible ones.

### Compatible versus incompatible observables

Two observables are **compatible** if $[A,B]=0$ — then ([§6.2](operators-spectral-theorem.ipynb)) they share an eigenbasis and *can* be
simultaneously definite — and **incompatible** if $[A,B]\ne0$, when no common eigenbasis exists,

```{math}
:label: eq-compatible
[A,B]=0\ \Longleftrightarrow\ \text{simultaneously diagonalizable}\ \Longleftrightarrow\ \text{can be jointly sharp in a common basis} .
```

Concretely: in $|{+}z\rangle$, where $\sigma_z$ is perfectly definite, $\langle\sigma_x\rangle=0$ and
$(\Delta\sigma_x)^2=1$ — *maximal* uncertainty. This is the algebra of the sequential experiment.

### The Robertson uncertainty relation — derived

For any two observables and any state,

```{math}
:label: eq-uncertainty
\Delta A\,\Delta B\ \ge\ \tfrac12\,\big|\langle[A,B]\rangle\big|,\qquad \Delta A=\sqrt{\langle A^2\rangle-\langle A\rangle^2} .
```

*Derivation (Exercise 3).* With the shifted operators $\delta A=A-\langle A\rangle$, $\delta B=B-
\langle B\rangle$, the variance is $(\Delta A)^2=\langle\delta A^2\rangle$. Apply **Cauchy–Schwarz**
([§6.1](complex-vector-spaces.ipynb)) to $|\delta A\,\psi\rangle$ and $|\delta B\,\psi\rangle$: $\langle\delta A^2\rangle\langle\delta
B^2\rangle\ge|\langle\delta A\,\delta B\rangle|^2$. Split $\delta A\,\delta B=\tfrac12\{\delta A,\delta
B\}+\tfrac12[A,B]$ into its Hermitian (anticommutator, real expectation) and anti-Hermitian
(commutator, imaginary expectation) parts; dropping the anticommutator term keeps $|\langle\delta A\,
\delta B\rangle|^2\ge\big(\tfrac12|\langle[A,B]\rangle|\big)^2$, and the square root is the relation.
It is **Cauchy–Schwarz plus non-commutation** — a theorem about operators, not instruments. (Keeping
the anticommutator term gives the stronger Schrödinger relation; a pointer, not developed.)

### Minimum-uncertainty states and complete sets

The inequality is **saturated** by special states,

```{math}
:label: eq-saturation
\text{for }(S_x,S_y):\quad |{+}z\rangle\ \text{gives}\ \Delta S_x\,\Delta S_y=\tfrac14=\tfrac12|\langle S_z\rangle| ,
```

the most "classical" allowed states (the Gaussian wave packet is their position–momentum analogue,
[§6.9](position-representation.ipynb)). And commuting observables can be measured together and labelled by joint eigenvalues; a maximal
commuting set — a **complete set of commuting observables** — labels states uniquely,

```{math}
:label: eq-cso
\{A,B,\dots\}\ \text{mutually commuting and maximal}\ \Longrightarrow\ \text{joint eigenvalues label states uniquely} ,
```

which is how quantum states are catalogued (the hydrogen atom's $n,\ell,m,m_s$, [§6.17](hydrogen-atom.ipynb)).

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from ecp import draw, validate

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = "#c1121f"

# Conventions: ℏ=1, so the spin operators are S = ½σ with eigenvalues ±½. Observables are
# Hermitian matrices; moments use numpy.vdot (conjugate-first, §6.1); commutators/anticommutators use the
# matrix-multiply @. The qubit is the running example; general checks use Hermitian matrices on ℂ³.

# the Pauli matrices (the SX/SY/SZ of §6.2), the canonical incompatible observables
SIGMA_X = np.array([[0, 1], [1, 0]], dtype=complex)
SIGMA_Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
SIGMA_Z = np.array([[1, 0], [0, -1]], dtype=complex)
ID2 = np.eye(2, dtype=complex)
# spin operators S = ½σ (ℏ = 1)
S_X, S_Y, S_Z = SIGMA_X / 2, SIGMA_Y / 2, SIGMA_Z / 2


def commutator(A, B):
    """The commutator $[A,B]=AB-BA$ {eq}`eq-pauli-algebra`.

    Computed with the matrix-multiply operator ``@``. It vanishes for compatible observables and is
    non-zero for incompatible ones; for Hermitian $A,B$ it is anti-Hermitian, so $\\langle[A,B]\\rangle$
    is purely imaginary — the fact that makes the uncertainty bound real.
    """
    return A @ B - B @ A


def anticommutator(A, B):
    """The anticommutator $\\{A,B\\}=AB+BA$ {eq}`eq-pauli-algebra`.

    Computed with ``@``. For Hermitian $A,B$ it is Hermitian, so $\\langle\\{A,B\\}\\rangle$ is real; it
    is the term dropped in the Robertson relation and kept in the stronger Schrödinger relation.
    """
    return A @ B + B @ A


def expectation(A, psi):
    """The expectation value $\\langle A\\rangle=\\langle\\psi|A|\\psi\\rangle$ (§6.5), via ``numpy.vdot``.

    Real because $A$ is Hermitian (``numpy.vdot(psi, A @ psi).real``).
    """
    return np.vdot(psi, A @ psi).real


def dispersion(A, psi):
    """The uncertainty $\\Delta A=\\sqrt{\\langle A^2\\rangle-\\langle A\\rangle^2}$ {eq}`eq-uncertainty` (§6.5).

    The square root of the variance — zero exactly when ``psi`` is an eigenstate of ``A`` (a definite
    value). A small negative variance from rounding is clipped to zero before the square root.
    """
    var = expectation(A @ A, psi) - expectation(A, psi) ** 2
    return np.sqrt(max(var, 0.0))


def robertson_rhs(A, B, psi):
    """The Robertson lower bound $\\tfrac12|\\langle[A,B]\\rangle|$ {eq}`eq-uncertainty`.

    Half the magnitude of the commutator's (purely imaginary) expectation value,
    ``0.5 * numpy.abs(numpy.vdot(psi, commutator(A, B) @ psi))`` — the floor under $\\Delta A\\,\\Delta B$.
    """
    return 0.5 * np.abs(np.vdot(psi, commutator(A, B) @ psi))

## Exercise 1 — The Pauli algebra

Verify the algebra of the three Pauli matrices: the commutation relations $[\sigma_x,
\sigma_y]=2i\sigma_z$ and its cyclic partners, the anticommutation $\{\sigma_i,\sigma_j\}=0$ for
$i \ne j$, the squares $\sigma_i^2=I$, and the product identity $\sigma_i\sigma_j=\delta_{ij}I+i
\varepsilon_{ijk}\sigma_k$ {eq}`eq-pauli`, {eq}`eq-pauli-algebra`.

1. Take the matrices `SIGMA_X`, `SIGMA_Y`, `SIGMA_Z` from the setup.
2. Compute the three commutators with the `commutator` helper and confirm with `numpy.allclose`
   that they equal `2j*SIGMA_Z`, `2j*SIGMA_X`, `2j*SIGMA_Y`.
3. Confirm the anticommutators `anticommutator(SIGMA_i, SIGMA_j)` vanish for $i\ne j$ and that
   each `SIGMA_i @ SIGMA_i` equals `ID2`.
4. Verify the product identity on a representative pair, e.g. `SIGMA_X @ SIGMA_Y` equals
   `1j*SIGMA_Z`. Note that $S=\tfrac12 \sigma$ turns the first relation into the angular-momentum
   algebra $[S_x,S_y]=iS_z$ ($\hbar=1$).

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.check(
    comm_xy and comm_yz and comm_zx and anticomm and squares and prod_identity,
    "the Pauli matrices satisfy the full algebra: [σx,σy]=2iσz (cyclic), {σi,σj}=0 for i≠j, σ²=I, and σiσj=δijI+iεijkσk",
)

## Exercise 2 — The commutator as the test for compatibility

Recall the sequential Stern–Gerlach destruction of [§6.4](stern-gerlach-qubit.ipynb). Show it is governed by the commutator:
$\sigma_z$ and $\sigma_x$ are **incompatible** ($[\sigma_z,\sigma_x]\ne0$), so a state perfectly
definite along $z$ is *maximally* uncertain along $x$; contrast a **compatible** pair, which can
be jointly sharp {eq}`eq-compatible`.

1. Compute $[\sigma_z,\sigma_x]$ with the `commutator` helper and confirm it is non-zero —
   incompatible.
2. In the $\sigma_z$-eigenstate $|{+}z\rangle=$`numpy.array([1,0])`, compute
   $\langle\sigma_x\rangle$ with the `expectation` helper and $(\Delta\sigma_x)^2$ with the
   `dispersion` helper squared.
3. Find $\langle\sigma_x\rangle=0$ and $(\Delta\sigma_x)^2=1$ — the maximum possible for an
   observable with eigenvalues $\pm1$ (no common eigenbasis).
4. Contrast with the compatible pair $(\sigma_z,\sigma_z^2)$: $[\sigma_z,\sigma_z^2]=0$, and both
   are perfectly definite in $|{+}z \rangle$. Frame: this is the sequential experiment,
   algebraically.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.close(
    [var_sx, exp_sx],
    [1.0, 0.0],
    "an incompatible observable is maximally uncertain in an eigenstate of its partner: σx has ⟨σx⟩=0 and (Δσx)²=1 in |+z⟩",
    atol=1e-12,
)
validate.check(
    incompatible and compatible and both_sharp,
    "the commutator is the test for compatibility: [σz,σx]≠0 (incompatible, cannot be jointly sharp) while [σz,σz²]=0 (compatible, both definite)",
)

## Exercise 3 — Deriving the uncertainty relation

Derive the Robertson uncertainty relation $\Delta A\,\Delta B\ge\tfrac12|\langle[A,B] \rangle|$
from the Cauchy–Schwarz inequality of [§6.1](complex-vector-spaces.ipynb), and verify each step numerically for random Hermitian
observables and a random state {eq}`eq-uncertainty`.

1. Build Hermitian $A,B$ on $\mathbb{C}^3$ and a unit state, and form the **shifted** operators
   $\delta A=A-\langle A\rangle I$ and $\delta B=B-\langle B\rangle I$, so $(\Delta A)^2=
   \langle\delta A^2\rangle$.
2. **Cauchy–Schwarz** ([§6.1](complex-vector-spaces.ipynb)) applied to $|\delta A\,\psi\rangle$ and $|\delta B\,\psi\rangle$ gives
   $\langle\delta A^2\rangle\langle\delta B^2\rangle\ge|\langle\delta A\, \delta B\rangle|^2$ —
   verify it (`numpy.vdot` for each inner product).
3. Split $\delta A\,\delta B= \tfrac12\{\delta A,\delta B\}+\tfrac12[A,B]$; confirm
   $\langle\{\delta A,\delta B\}\rangle$ is real and $\langle[A,B]\rangle$ is purely imaginary, so
   $|\langle\delta A\,\delta B\rangle|^2=\big(\tfrac12 \langle\{\delta A,\delta
   B\}\rangle\big)^2+\big(\tfrac12|\langle[A,B]\rangle|\big)^2$.
4. Dropping the (non-negative) anticommutator term leaves $\Delta A\,\Delta
   B\ge\tfrac12|\langle[A,B]\rangle|$ — the `robertson_rhs` helper. Frame: the uncertainty
   principle is Cauchy–Schwarz plus non-commutation.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
# verify the derivation on many random cases
rng_v = np.random.default_rng(20)
all_hold = True
for _ in range(3000):
    Ma = rng_v.standard_normal((3, 3)) + 1j * rng_v.standard_normal((3, 3))
    Mb = rng_v.standard_normal((3, 3)) + 1j * rng_v.standard_normal((3, 3))
    Ar, Br = Ma + Ma.conj().T, Mb + Mb.conj().T
    p = rng_v.standard_normal(3) + 1j * rng_v.standard_normal(3)
    p = p / np.linalg.norm(p)
    if dispersion(Ar, p) * dispersion(Br, p) < robertson_rhs(Ar, Br, p) - 1e-10:
        all_hold = False
validate.check(
    decomposition and all_hold,
    "ΔA·ΔB ≥ ½|⟨[A,B]⟩| follows from Cauchy–Schwarz: |⟨δAδB⟩|² splits into anticommutator + commutator squares, and the relation holds for all tested observables and states",
)

## Exercise 4 — The spin uncertainty relation, verified and visualized

Specialize the relation to the spin components: verify $\Delta S_x\,\Delta S_y\ge \tfrac12|\langle
S_z\rangle|$ across a family of qubit states, and visualize how the uncertainty product rides
above the state-dependent bound {eq}`eq-uncertainty`.

1. Parametrize qubit states on the Bloch sphere, $|\psi(\theta,\varphi)\rangle=
   \cos\tfrac{\theta}{2}|{+}z\rangle+e^{i\varphi}\sin\tfrac{\theta}{2}|{-}z\rangle$ (a light
   preview of [§6.8](bloch-sphere-entanglement.ipynb)), sweeping $\theta$ at a fixed generic $\varphi$.
2. For each state compute $\Delta S_x$, $\Delta S_y$ with the `dispersion` helper and the bound
   $\tfrac12|\langle S_z\rangle|$ with the `robertson_rhs` helper (which equals it, since
   $[S_x,S_y]=iS_z$).
3. Confirm $\Delta S_x\,\Delta S_y\ge$ the bound everywhere.
4. Plot the product and the bound against $\theta$. Frame: the bound is *state-dependent* (through
   $\langle S_z\rangle$) — uncertainty is a trade-off set by the state, not a fixed number.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.check(
    holds_everywhere,
    "the spin-component uncertainty relation ΔSx·ΔSy ≥ ½|⟨Sz⟩| holds across the whole Bloch-sphere family of states",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 5 — Minimum-uncertainty states

The relation is an inequality, but for special states it becomes an equality. Show that the
$\sigma_z$-eigenstate $|{+}z\rangle$ **saturates** the spin uncertainty relation, $\Delta S_x\,
\Delta S_y=\tfrac12|\langle S_z\rangle|=\tfrac14$ {eq}`eq-saturation`.

1. For $|{+}z\rangle$, compute $\Delta S_x$ and $\Delta S_y$ with the `dispersion` helper and
   their product.
2. Compute the bound $\tfrac12|\langle S_z\rangle|$ with the `robertson_rhs` helper.
3. Confirm product and bound are equal ($\tfrac14=\tfrac14$) — a minimum-uncertainty state.
4. Note that a generic state (e.g. $\theta=1,\varphi=1$) does **not** saturate, leaving slack.
   Frame: minimum-uncertainty states are the most "classical" allowed; the Gaussian wave packet is
   their position–momentum analogue ([§6.9](position-representation.ipynb)).

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    prod_upz,
    bound_upz,
    "|+z⟩ saturates the uncertainty relation, ΔSx·ΔSy = ½|⟨Sz⟩| = ¼ — a minimum-uncertainty state",
    rtol=1e-12,
)

## Exercise 6 — Compatible observables and a complete set

When observables commute the trade-off disappears: they can be simultaneously sharp, and their
joint eigenvalues label states. Verify that $S_z$ and $S_z^2$ commute and are both definite in
$|{+}z\rangle$, and explain how a maximal commuting set labels states uniquely
{eq}`eq-compatible`, {eq}`eq-cso`.

1. Compute $[S_z,S_z^2]$ with the `commutator` helper (`S_Z @ S_Z` for $S_z^2$) and confirm it is
   the zero matrix (`numpy.allclose`).
2. Confirm both $\Delta S_z$ and $\Delta S_z^2$ are zero in $|{+}z\rangle$ with the `dispersion`
   helper — both perfectly definite.
3. Note that the commuting bound $\tfrac12|\langle[S_z,S_z^2]\rangle|=0$, so the relation is
   trivially satisfied with no trade-off.
4. Reason in prose: joint eigenstates of commuting observables carry joint eigenvalue labels, and
   a **complete set of commuting observables** labels every state uniquely — the way the hydrogen
   atom's states are tagged by $(n,\ell,m,m_s)$ in [§6.17](hydrogen-atom.ipynb). Frame: how quantum states are catalogued.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.check(
    compat and np.isclose(dSz, 0) and np.isclose(dSz2, 0),
    "compatible observables ([Sz,Sz²]=0) can be simultaneously definite (both sharp in |+z⟩) and jointly label states",
)

## Exercise 7 — The general uncertainty relation in a larger space *(student)*

The relation is universal — it holds for *any* pair of observables on *any* space, not just spins.
For two random Hermitian observables on $\mathbb{C}^3$ and a random state, verify Robertson's
inequality and measure how far the state is from saturating it {eq}`eq-uncertainty`.

1. Build Hermitian $A,B$ on $\mathbb{C}^3$ (symmetrize random complex matrices, `M + M.conj().T`)
   and a normalized random state.
2. Compute $\Delta A$, $\Delta B$ with the `dispersion` helper and the bound with `robertson_rhs`.
3. Confirm $\Delta A\,\Delta B\ge\tfrac12| \langle[A,B]\rangle|$ and report the **saturation
   ratio** $\tfrac12|\langle[A,B]\rangle|/(\Delta A\, \Delta B)\in[0,1]$.
4. Observe that the ratio is usually well below $1$ — the bound is loose unless the state is
   special. Frame: the inequality is universal but rarely tight.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.check(
    lhs >= rhs - 1e-12 and 0 <= ratio <= 1 + 1e-12,
    "the Robertson uncertainty relation holds for an arbitrary pair of Hermitian observables on ℂ³, with a saturation ratio in [0,1]",
)

## Exercise 8 — The uncertainty principle is an algebraic fact

The strangeness of the sequential Stern–Gerlach experiment was never about disturbing delicate atoms.
It was the **algebra**. Two observables that fail to commute cannot both be sharp in any state, and
*how* sharply they can be co-defined is bounded below by their commutator: $\Delta A\,\Delta B\ge
\tfrac12|\langle[A,B]\rangle|$. The Pauli matrices made this concrete — the three spin components
pairwise anticommute and fail to commute, so definiteness along one axis forces maximal uncertainty
along the others — and **Cauchy–Schwarz**, the same inequality that bounded amplitudes in [§6.1](complex-vector-spaces.ipynb), made it
general. Minimum-uncertainty states saturate the bound; commuting observables escape it entirely and
can be jointly sharp, labelling states by their joint eigenvalues.

**This exercise (synthesis).** No new computation: the theorem is the result. "You cannot know both at
once" sounds like a limitation imposed on us by clumsy instruments. It is the opposite — a fact about
the observables themselves, true whether or not anyone is looking, and we derived it in three lines
from the geometry of the Hilbert space. The next notebook ([§6.7](time-evolution.ipynb)) finally sets the state in motion: the
dynamics postulate worked out, the Hamiltonian generating unitary evolution, and the spin precessing
on the very Bloch sphere this uncertainty trade-off lives on.

## Notebook summary

The algebra behind the incompatibility that Stern–Gerlach showed and the variance measured — and the
uncertainty principle as its theorem.

- **The Pauli algebra** {eq}`eq-pauli`, {eq}`eq-pauli-algebra`: $[\sigma_x,\sigma_y]=2i\sigma_z$
  (cyclic), $\{\sigma_i,\sigma_j\}=0$, $\sigma_i^2=I$, $\sigma_i\sigma_j=\delta_{ij}I+i\varepsilon_{ijk}
  \sigma_k$ — equivalently $[S_x,S_y]=iS_z$, the angular-momentum algebra.
- **Compatibility** {eq}`eq-compatible`: $[A,B]=0$ means jointly sharp; $[A,B]\ne0$ means not — and
  $\sigma_x$ is *maximally* uncertain ($(\Delta\sigma_x)^2=1$) in a $\sigma_z$-eigenstate.
- **The Robertson relation, derived** {eq}`eq-uncertainty`: $\Delta A\,\Delta B\ge\tfrac12|\langle[A,B]
  \rangle|$, from Cauchy–Schwarz ([§6.1](complex-vector-spaces.ipynb)) plus the commutator/anticommutator split — a theorem about
  operators, true of every state, *not* measurement disturbance.
- **Minimum-uncertainty states** {eq}`eq-saturation`: $|{+}z\rangle$ saturates $\Delta S_x\,\Delta S_y
  =\tfrac14=\tfrac12|\langle S_z\rangle|$ — the most classical states allowed.
- **Complete sets** {eq}`eq-cso`: commuting observables can be jointly sharp and label states by joint
  eigenvalues (hydrogen's $n,\ell,m,m_s$, [§6.17](hydrogen-atom.ipynb)).

The uncertainty principle is Cauchy–Schwarz plus non-commutation. The commutator is literally the
floor on how much two observables must trade off — an algebraic fact, true whether or not anyone looks.

## Outlook

- **Time evolution ([§6.7](time-evolution.ipynb))**: the Hamiltonian as generator, $U(t)=e^{-iHt/\hbar}$, spin precession and
  Rabi oscillations — the state set in motion.
- **The Bloch sphere ([§6.8](bloch-sphere-entanglement.ipynb))**: the geometry of qubit states, where this uncertainty trade-off is
  visible.
- **Position and momentum ([§6.9](position-representation.ipynb))**: $[x,p]=i\hbar$ gives $\Delta x\,\Delta p\ge\hbar/2$, with the
  Gaussian as the minimum-uncertainty state.
- **The angular-momentum algebra generalized ([§6.14](angular-momentum-algebra.ipynb))**: $[J_i,J_j]=i\hbar\varepsilon_{ijk}J_k$, the
  full theory of spin and orbital angular momentum, and complete sets of commuting observables ([§6.17](hydrogen-atom.ipynb)).
- **Cross-reference** [§6.1](complex-vector-spaces.ipynb) (Cauchy–Schwarz), [§6.2](operators-spectral-theorem.ipynb) (commutators / common eigenbasis), [§6.4](stern-gerlach-qubit.ipynb) (the
  experiment), [§6.5](postulates.ipynb) (variance), and forward to [§6.7](time-evolution.ipynb), [§6.8](bloch-sphere-entanglement.ipynb), [§6.9](position-representation.ipynb), [§6.14](angular-momentum-algebra.ipynb), [§6.17](hydrogen-atom.ipynb).

In [ ]:
from ecp.style import footer

footer()